# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR² dataset using the `mlcroissant` library.

### Dataset Source

The dataset is described by a Croissant schema available at the following URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure the mlcroissant package is installed (uncomment if running in Colab or a new environment)
!pip install mlcroissant

## 1. Data Loading

Load the Croissant dataset metadata and make a basic inspection.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset title:', metadata.name)
print('Description:', metadata.description)
print('Published:', getattr(metadata, 'datePublished', ''))
print('License:', getattr(metadata, 'license', ''))

## 2. Data Overview

Review the available record sets (tables), and for each, list their fields and columns using `@id` fields.

> All entities (record sets, fields, columns) are referenced by their `@id`.

Let's enumerate all record sets and their available fields and columns:

In [ ]:
# List all record sets with their @id and schema
print('Record Sets in this dataset:')
record_sets = []
for rs in dataset.record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '')}")
    print(f"  Fields:")
    for fld in rs.get('field', []):
        print(f"    - @id: {fld['@id']}, name: {fld.get('name', '')}, dataType: {fld.get('dataType', '')}")
    print(f"  Columns:")
    for col in rs.get('column', []):
        print(f"    - @id: {col['@id']}, name: {col.get('name', '')}, dataType: {col.get('dataType', '')}")
    record_sets.append(rs['@id'])

# Show a sample record from each record set, referenced by @id
for rs_id in record_sets:
    print(f"\nSample record from record set @id: {rs_id}")
    recs = dataset.records(record_set=rs_id)
    try:
        print(next(recs))
    except StopIteration:
        print('No records found for this record set.')

## 3. Data Extraction

Load the data for the record set(s) of interest into pandas DataFrames, using `@id` references throughout.

For demonstration, we will extract all record sets. Replace the items in `selected_record_sets` with the specific `@id`s you wish to analyze further.

In [ ]:
# List of all record set @ids
selected_record_sets = record_sets.copy()
dataframes = {}

for rs_id in selected_record_sets:
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df
    print(f"\nLoaded {len(df)} records from record set @id: {rs_id}")
    print(f"Columns (field @id): {list(df.columns)}")
    print(df.head(2)) if not df.empty else print('No records to display.')

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field and a group field by their `@id` for further analysis. We'll show basic filtering, normalization, and grouping actions by `@id`.

In [ ]:
# Example: identify a main table for EDA (update this @id based on overview prints above). Replace these variables accordingly:
# If you are unsure, run the previous cell, inspect the outputs, and copy-paste the @id for a main clinical table.
main_rs_id = selected_record_sets[0]  # change if needed
df = dataframes[main_rs_id]

if df.empty:
    print(f'No records in record set @id: {main_rs_id}')
else:
    # Choose numeric columns - here you should set these to the `@id` strings corresponding to numeric fields
    print('Available fields for analysis:', list(df.columns))

    # Example: Assume we have '@id': 'age' and '@id': 'sex' for demonstration (replace with actual @ids/field names)
    numeric_field_id = None  # e.g. 'age' (use @id string from above)
    group_field_id = None    # e.g. 'sex' (use @id string from above)

    # Try to pick a numeric field
    for col in df.columns:
        # Detect possible integer/float columns by dtype
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Try a non-numeric grouping field
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

    if not numeric_field_id:
        print('No numeric field found for EDA! Please adjust the variables.')
    else:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If there's a suitable group field, show group means
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print('No suitable group field found for grouping analysis.')

## 5. Visualization

Let's visualize the distribution of our numeric field, and show a grouped bar for the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=10, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id], palette='Set2')
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Load and inspect a biomedical dataset described by a Croissant schema using its `@id` references
- Enumerate available record sets and fields
- Load record sets into pandas DataFrames by `@id`
- Perform exploratory data analysis (EDA) using field and record set `@id`s
- Visualize data distributions and groupwise statistics

This framework supports reproducible and robust FAIR data science workflows for structured biomedical datasets.